# 3.4 Lab: Multi-Latent Attention (MLA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.4_mla/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.4_mla/lab.ipynb)

This lab demonstrates MLA's core mechanics:
1. Low-rank KV projection (down-project and up-project)
2. Compressed KV cache vs full cache
3. Memory comparison: MHA vs GQA vs MLA
4. The matrix absorption trick (compute savings)

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## Experiment 1: Low-Rank KV Projection

MLA compresses the hidden state into a latent vector before caching.
We simulate the down-projection and verify reconstruction quality.

In [ ]:
# --- Model parameters (DeepSeek-V2 scale) ---
d = 5120          # hidden dimension
n_h = 128         # number of attention heads
d_h = 128         # per-head dimension
d_c = 512         # KV compression dimension (latent size)
d_h_R = 64        # decoupled RoPE dimension

# --- Simulate a single token's hidden state ---
h_t = np.random.randn(d).astype(np.float32)

# --- Down-projection: compress h_t to latent ---
# W_DKV shape: [d_c, d] -- projects from full hidden to latent
W_DKV = np.random.randn(d_c, d).astype(np.float32) * 0.01
c_kv = W_DKV @ h_t  # latent vector, shape [d_c]

print(f"Hidden state shape: {h_t.shape} ({h_t.nbytes:,} bytes)")
print(f"Latent vector shape: {c_kv.shape} ({c_kv.nbytes:,} bytes)")
print(f"Compression ratio: {h_t.nbytes / c_kv.nbytes:.1f}x")

## Experiment 2: Up-Projection Recovery

From the cached latent, recover full keys and values via learned up-projections.

In [ ]:
# --- Up-projection matrices ---
# W_UK shape: [n_h * d_h, d_c] -- recovers keys from latent
# W_UV shape: [n_h * d_h, d_c] -- recovers values from latent
full_kv_dim = n_h * d_h  # 128 * 128 = 16,384

W_UK = np.random.randn(full_kv_dim, d_c).astype(np.float32) * 0.01
W_UV = np.random.randn(full_kv_dim, d_c).astype(np.float32) * 0.01

# --- Recover full keys and values ---
keys_recovered = W_UK @ c_kv    # shape [n_h * d_h]
values_recovered = W_UV @ c_kv  # shape [n_h * d_h]

# --- Compare: standard projection (no compression) ---
W_K_standard = np.random.randn(full_kv_dim, d).astype(np.float32) * 0.01
keys_standard = W_K_standard @ h_t

print(f"Standard key dim: {keys_standard.shape} (cached per token)")
print(f"MLA latent dim:   {c_kv.shape} (cached per token)")
print(f"Recovered key dim: {keys_recovered.shape} (computed on-the-fly)")
print(f"\nCache savings: {keys_standard.nbytes * 2 / c_kv.nbytes:.1f}x (K+V vs latent)")

## Experiment 3: Memory Comparison (MHA vs GQA vs MLA)

Compare KV cache memory across attention variants for practical serving scenarios.

In [ ]:
# --- Parameters ---
seq_lengths = [1024, 2048, 4096, 8192, 16384, 32768]
batch_size = 32
n_layers = 60
bytes_per_elem = 2  # FP16

# --- Cache per token per layer (elements) ---
mha_per_token = 2 * n_h * d_h                  # 32,768
gqa8_per_token = 2 * 8 * d_h                   # 2,048
gqa4_per_token = 2 * 4 * d_h                   # 1,024
mla_per_token = d_c + d_h_R                    # 576

# --- Compute total cache (GB) for each seq length ---
def cache_gb(per_token_elems, seq_len):
    return per_token_elems * n_layers * bytes_per_elem * batch_size * seq_len / (1024**3)

mha_gb = [cache_gb(mha_per_token, s) for s in seq_lengths]
gqa8_gb = [cache_gb(gqa8_per_token, s) for s in seq_lengths]
gqa4_gb = [cache_gb(gqa4_per_token, s) for s in seq_lengths]
mla_gb = [cache_gb(mla_per_token, s) for s in seq_lengths]

# --- Plot ---
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
seq_k = [s // 1024 for s in seq_lengths]  # x-axis in K tokens

ax.plot(seq_k, mha_gb, 'o-', color='#dc2626', linewidth=2, label=f'MHA ({mha_per_token:,} elem/tok/layer)')
ax.plot(seq_k, gqa8_gb, 's-', color='#2563eb', linewidth=2, label=f'GQA-8 ({gqa8_per_token:,} elem/tok/layer)')
ax.plot(seq_k, gqa4_gb, '^-', color='#7c3aed', linewidth=2, label=f'GQA-4 ({gqa4_per_token:,} elem/tok/layer)')
ax.plot(seq_k, mla_gb, 'D-', color='#059669', linewidth=2, label=f'MLA ({mla_per_token} elem/tok/layer)')

# GPU memory reference lines
ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='A100 80GB')
ax.axhline(y=24, color='gray', linestyle=':', alpha=0.5, label='A10G 24GB')

ax.set_xlabel('Sequence Length (K tokens)', fontsize=12)
ax.set_ylabel('KV Cache Size (GB)', fontsize=12)
ax.set_title(f'KV Cache Memory: MHA vs GQA vs MLA\n(batch={batch_size}, layers={n_layers}, FP16)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 120)
plt.tight_layout()
plt.savefig('mla_memory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: mla_memory_comparison.png")

## Experiment 4: The Matrix Absorption Trick

Demonstrate the compute savings from absorbing W_UK into the query projection.

In [ ]:
# --- Simulate attention score computation for S cached tokens ---
S = 4096  # sequence length (cached tokens)

# Generate cached latents for S tokens
cached_latents = np.random.randn(S, d_c).astype(np.float32) * 0.01

# Single query head
q_head = np.random.randn(d_h).astype(np.float32)

# Up-projection for one head: [d_h, d_c]
W_UK_head = np.random.randn(d_h, d_c).astype(np.float32) * 0.01

# --- Method 1: Naive (decompress each cached token) ---
# For each token j: score_j = q^T @ (W_UK @ c_j)
scores_naive = np.zeros(S)
for j in range(S):
    k_j = W_UK_head @ cached_latents[j]   # [d_h, d_c] @ [d_c] -> [d_h]
    scores_naive[j] = q_head @ k_j         # [d_h] @ [d_h] -> scalar

# --- Method 2: Absorbed (compress query once) ---
# q_compressed = W_UK^T @ q, then score_j = q_compressed^T @ c_j
q_compressed = W_UK_head.T @ q_head        # [d_c, d_h] @ [d_h] -> [d_c]
scores_absorbed = cached_latents @ q_compressed  # [S, d_c] @ [d_c] -> [S]

# --- Verify equivalence ---
max_diff = np.max(np.abs(scores_naive - scores_absorbed))
print(f"Max difference between methods: {max_diff:.2e} (numerical noise)")
print(f"Methods are equivalent: {max_diff < 1e-3}")

# --- FLOP comparison ---
flops_naive = S * d_h * d_c + S * d_h      # S matmuls + S dots
flops_absorbed = d_c * d_h + S * d_c       # 1 matmul + S dots
print(f"\nNaive FLOPs:    {flops_naive:>12,}")
print(f"Absorbed FLOPs: {flops_absorbed:>12,}")
print(f"Speedup:        {flops_naive / flops_absorbed:.0f}x")

## Experiment 5: Equivalent GQA Groups

MLA's cache size can be expressed as an equivalent number of GQA groups.
This shows how aggressive the compression is.

In [ ]:
# --- Sweep d_c values and compute equivalent GQA groups ---
d_c_values = [128, 256, 512, 768, 1024, 1536, 2048]
equiv_groups = []
cache_reductions_vs_mha = []

for dc in d_c_values:
    # MLA cache elements per token per layer
    mla_elems = dc + d_h_R
    # Equivalent GQA groups: mla_elems = 2 * n_groups * d_h
    n_groups_equiv = mla_elems / (2 * d_h)
    equiv_groups.append(n_groups_equiv)
    # Reduction vs MHA
    mha_elems = 2 * n_h * d_h
    cache_reductions_vs_mha.append(mha_elems / mla_elems)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: equivalent GQA groups
ax1.bar(range(len(d_c_values)), equiv_groups, color='#2563eb', alpha=0.8)
ax1.set_xticks(range(len(d_c_values)))
ax1.set_xticklabels([str(dc) for dc in d_c_values])
ax1.set_xlabel('d_c (MLA latent dimension)', fontsize=11)
ax1.set_ylabel('Equivalent GQA Groups', fontsize=11)
ax1.set_title('MLA Compression as Equivalent GQA Groups', fontsize=12)
ax1.axhline(y=8, color='#dc2626', linestyle='--', alpha=0.7, label='Llama GQA-8')
ax1.axhline(y=1, color='#059669', linestyle='--', alpha=0.7, label='MQA (1 group)')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Right: cache reduction vs MHA
ax2.bar(range(len(d_c_values)), cache_reductions_vs_mha, color='#059669', alpha=0.8)
ax2.set_xticks(range(len(d_c_values)))
ax2.set_xticklabels([str(dc) for dc in d_c_values])
ax2.set_xlabel('d_c (MLA latent dimension)', fontsize=11)
ax2.set_ylabel('Cache Reduction vs MHA (x)', fontsize=11)
ax2.set_title('KV Cache Reduction Factor vs MHA', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

# Mark DeepSeek-V2 choice
for ax in [ax1, ax2]:
    ax.axvline(x=2, color='#f59e0b', linestyle='-', alpha=0.5, linewidth=8)
    ax.annotate('DeepSeek-V2\n(d_c=512)', xy=(2, ax.get_ylim()[1]*0.85),
                ha='center', fontsize=9, color='#92400e')

plt.tight_layout()
plt.savefig('mla_equivalent_gqa.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: mla_equivalent_gqa.png")

## Key Takeaways

1. **MLA compresses K+V jointly** into a single latent vector of dimension d_c, achieving 56.9x cache reduction vs MHA
2. **Matrix absorption** lets attention operate directly in compressed space (124x FLOP savings at seq_len=4096)
3. **Decoupled RoPE** separates positional encoding into a small key to preserve absorption compatibility
4. **Quality preserved**: DeepSeek-V2 matches MHA on MMLU/BBH despite 93.3% smaller cache
5. **Practical impact**: 5.76x higher generation throughput from enabling larger batch sizes